# 9장 머신러닝 시작: 날씨 회귀

이 노트북은 웹 교재 [9장 머신러닝 시작: 날씨 회귀](https://pidocs.aifusionlab.org/ai-python/ch09/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/seoul_weather_2021_2025.csv
!ls *.csv

## 9.2 날씨 자료의 특징과 레이블

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch09/9-2/)

In [ ]:
import pandas as pd

df = pd.read_csv("seoul_weather_2021_2025.csv", parse_dates=["date"])
print(df.shape)
print(df.head(3))

In [ ]:
df["target_temp"] = df["temp_mean"].shift(-1)
print(df[["date", "temp_mean", "target_temp"]].tail(3))

In [ ]:
import numpy as np

day = df["date"].dt.day_of_year
df["doy_sin"] = np.sin(2 * np.pi * day / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * day / 365.25)
print(df.loc[[0, 90, 181, 273], ["date", "doy_sin", "doy_cos"]].round({"doy_sin": 2, "doy_cos": 2}))

In [ ]:
import matplotlib.pyplot as plt

one = df[df["date"].dt.year == 2021]
plt.figure(figsize=(5, 4.5))
plt.scatter(one["doy_cos"], one["doy_sin"], c=one["temp_mean"], cmap="coolwarm", s=12)
plt.colorbar(label="temp_mean (C)")
plt.xlabel("doy_cos")
plt.ylabel("doy_sin")
plt.axis("equal")
plt.show()

## 9.3 학습 데이터와 평가 데이터의 분할

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch09/9-3/)

In [ ]:
data = df.dropna(subset=["target_temp"])
features = ["temp_mean", "temp_max", "temp_min", "humidity",
            "precip_mm", "wind_max", "sunshine_h", "doy_sin", "doy_cos"]

train = data[data["date"] < "2025-01-01"]
test = data[data["date"] >= "2025-01-01"]
print(len(train), len(test))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))
plt.plot(train["date"], train["temp_mean"], lw=0.8, label="train (2021-2024)")
plt.plot(test["date"], test["temp_mean"], lw=0.8, label="test (2025)")
plt.ylabel("temperature (C)")
plt.legend()
plt.show()

## 9.4 선형회귀를 이용한 다음 날 기온 예측

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch09/9-4/)

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(train[features], train["target_temp"])

coef = pd.Series(model.coef_, index=features).round(3)
print(coef)
print("절편:", round(model.intercept_, 3))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3.5))
coef.sort_values().plot.barh()
plt.axvline(0, color="gray", lw=1)
plt.xlabel("coefficient")
plt.show()

In [ ]:
pred = model.predict(test[features])
result = pd.DataFrame({"date": test["date"], "actual": test["target_temp"], "pred": pred.round(1)})
print(result.head(5))

## 9.5 회귀 모델 평가

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch09/9-5/)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(test["target_temp"], pred)
rmse = mean_squared_error(test["target_temp"], pred) ** 0.5
r2 = r2_score(test["target_temp"], pred)
print(f"MAE  {mae:.2f}")
print(f"RMSE {rmse:.2f}")
print(f"R2   {r2:.3f}")

In [ ]:
base = test["temp_mean"]          # 오늘 평균기온을 그대로 내일 예측값으로 사용
base_mae = mean_absolute_error(test["target_temp"], base)
print(f"기준선 MAE {base_mae:.2f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3.5))
plt.plot(result["date"], result["actual"], lw=1, label="actual")
plt.plot(result["date"], result["pred"], lw=1, label="pred")
plt.ylabel("next-day mean temperature (C)")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(4.5, 4.5))
plt.scatter(result["actual"], result["pred"], s=8)
plt.plot([-15, 35], [-15, 35], color="gray", lw=1)   # 예측값 = 실제값인 대각선
plt.xlabel("actual (C)")
plt.ylabel("pred (C)")
plt.show()

In [ ]:
result["error"] = (result["actual"] - result["pred"]).abs()
print(result.sort_values("error", ascending=False).head(3))